In [4]:
%%capture
!pip install tensorboard nltk transformers datasets sacrebleu accelerate evaluate
!pip install git+https://github.com/csebuetnlp/normalizer


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import shutil, os

for path in ["/content/drive/MyDrive/finetuned_mbart50", "/content/drive/MyDrive/model_checkpoints.zip"]:
    if os.path.exists(path):
        (shutil.rmtree if os.path.isdir(path) else os.remove)(path)
        print(f"Deleted {path}")
    else:
        print(f"Not found: {path}")


Not found: /content/drive/MyDrive/finetuned_mbart50
Not found: /content/drive/MyDrive/model_checkpoints.zip


In [7]:
%%capture
import torch, numpy as np, pandas as pd
from datasets import Dataset
from normalizer import normalize
from transformers import (
    MBart50TokenizerFast,
    MBartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
    TrainerCallback,
)
import nltk
from transformers.integrations import TensorBoardCallback

nltk.download('punkt')
nltk.download('punkt_tab')

import transformers
print(f'Transformers version: {transformers.__version__}')


In [8]:
# mBART-50 language codes
SRC_LANG = 'bn_IN'   # Bengali
TGT_LANG = 'en_XX'   # English
MODEL_NAME = 'facebook/mbart-large-50-many-to-many-mmt'

tokenizer = MBart50TokenizerFast.from_pretrained(
    MODEL_NAME,
    src_lang=SRC_LANG,
    tgt_lang=TGT_LANG
)

model = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)

print(f'Model loaded: {MODEL_NAME}')
print(f'Vocab size  : {tokenizer.vocab_size}')
print(f'bn_IN id    : {tokenizer.lang_code_to_id[SRC_LANG]}')
print(f'en_XX id    : {tokenizer.lang_code_to_id[TGT_LANG]}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Model loaded: facebook/mbart-large-50-many-to-many-mmt
Vocab size  : 250054
bn_IN id    : 250028
en_XX id    : 250004


In [9]:
# Freeze all encoder layers initially
for param in model.model.encoder.parameters():
    param.requires_grad = False
print('All encoder layers frozen initially.')


All encoder layers frozen initially.


In [10]:
class GradualUnfreezingCallback(TrainerCallback):
    """
    Gradually unfreeze encoder layers during training.
    unfreeze_schedule: dict mapping epoch -> layers to unfreeze (int or 'all')
    """
    def __init__(self, model, unfreeze_schedule):
        self.model = model
        self.unfreeze_schedule = unfreeze_schedule
        self.encoder_layers = model.model.encoder.layers
        self.total_layers = len(self.encoder_layers)

    def on_epoch_begin(self, args, state, control, **kwargs):
        trainer = kwargs.get('trainer', None)
        if trainer is None:
            return
        epoch = int(state.epoch)
        if epoch not in self.unfreeze_schedule:
            return
        n = self.unfreeze_schedule[epoch]
        start = 0 if n == 'all' else max(0, self.total_layers - n)
        print(f'\n Unfreezing encoder layers {start} to {self.total_layers - 1} at epoch {epoch}')
        for layer in self.encoder_layers[start:]:
            for param in layer.parameters():
                param.requires_grad = True
        trainer.create_optimizer()  # Rebuild optimizer with new params
        return control


In [11]:
tensorboard_callback = TensorBoardCallback()


In [12]:
max_length = 128

def tokenize_and_create_dataset(tokenizer, data_df, max_length=128, normalize_source=True):
    source_texts = (
        [normalize(str(t)) for t in data_df['bangla_speech']]
        if normalize_source else [str(t) for t in data_df['bangla_speech']]
    )
    target_texts = [str(t) for t in data_df['english_speech']]

    # Tokenize source (tokenizer uses src_lang='bn_IN' set at load time)
    encodings = tokenizer(
        source_texts,
        truncation=True, max_length=max_length, padding=False
    )

    # CRITICAL: use text_target= so en_XX token is prepended to labels
    decodings = tokenizer(
        text_target=target_texts,
        truncation=True, max_length=max_length, padding=False
    )

    labels = [
        [-100 if tok == tokenizer.pad_token_id else tok for tok in ids]
        for ids in decodings['input_ids']
    ]

    # Debug: show first 3 tokenized examples
    print('\n=== First 3 tokenized examples ===')
    for i in range(min(3, len(encodings['input_ids']))):
        inp = tokenizer.decode(encodings['input_ids'][i], skip_special_tokens=True)
        lbl_ids = [t if t != -100 else tokenizer.pad_token_id for t in labels[i]]
        lbl = tokenizer.decode(lbl_ids, skip_special_tokens=True)
        print(f'  [{i}] Input: {inp}')
        print(f'       Label: {lbl}')

    return Dataset.from_dict({
        'input_ids':      encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels':         labels,
    })


data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    padding='max_length', max_length=max_length, label_pad_token_id=-100
)


In [13]:
csv_path = '/content/all_train_dataset.csv'
train_df = pd.read_csv(csv_path)
train_df['bangla_speech']  = train_df['bangla_speech'].fillna('').astype(str)
train_df['english_speech'] = train_df['english_speech'].fillna('').astype(str)
print(train_df.head())
print(f'Shape: {train_df.shape}')


                          bangla_speech                        english_speech  \
0                      লকট মোর প্রিয় ফল           Jamrul is my favorite fruit   
1  যত হারাহারি সম্ভব বিয়ার ব্যবস্থা গরো  Arrange marriage as soon as possible   
2   যত তাড়াতাড়ি সম্ভব বিয়ার ব্যবস্থা কর  Arrange marriage as soon as possible   
3  যত তাড়াতাড়ি সম্ভব বিয়ার ব্যবস্থা করো  Arrange marriage as soon as possible   
4      যত জলদি সম্ভব বিয়ার ব্যবস্থা খরো  Arrange marriage as soon as possible   

      dialect  
0   Barishal   
1  Chittagong  
2  Mymensingh  
3    Noakhali  
4      Sylhet  
Shape: (9374, 3)


In [14]:
def load_translation_dataset(csv_path, bangla_col):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    df = df[[bangla_col.strip(), 'english_speech']]
    return df.rename(columns={bangla_col.strip(): 'bangla_speech'})


In [15]:
BASE = '/content/drive/MyDrive/dataset_translated/'

barishal_valid_df    = load_translation_dataset(BASE + 'validation/barisal_validation.csv',  'barishal_bangla_speech ')
chittagong_valid_df  = load_translation_dataset(BASE + 'validation/chittagong_validation.csv', 'chittagong_bangla_speech ')
mymensingh_valid_df  = load_translation_dataset(BASE + 'validation/mymensingh_validation.csv', 'mymensingh_bangla_speech ')
noakhali_valid_df    = load_translation_dataset(BASE + 'validation/noakhali_validation.csv',   'noakhali_bangla_speech ')
sylhet_valid_df      = load_translation_dataset(BASE + 'validation/sylhet_validation.csv',     'sylhet_bangla_speech ')
standard_valid_df    = load_translation_dataset(BASE + 'validation/sylhet_validation.csv',     'bangla_speech')

barishal_test_csv    = BASE + 'test/barisal_test.csv'
chittagong_test_csv  = BASE + 'test/chittagong_test.csv'
mymensingh_test_csv  = BASE + 'test/mymensingh_test.csv'
noakhali_test_csv    = BASE + 'test/noakhali_test.csv'
sylhet_test_csv      = BASE + 'test/sylhet_test.csv'

barishal_test_df    = load_translation_dataset(barishal_test_csv,   'barishal_bangla_speech ')
chittagong_test_df  = load_translation_dataset(chittagong_test_csv, 'chittagong_bangla_speech ')
mymensingh_test_df  = load_translation_dataset(mymensingh_test_csv, 'mymensingh_bangla_speech ')
noakhali_test_df    = load_translation_dataset(noakhali_test_csv,   'noakhali_bangla_speech ')
sylhet_test_df      = load_translation_dataset(sylhet_test_csv,     'sylhet_bangla_speech ')
standard_test_df    = load_translation_dataset(sylhet_test_csv,     'bangla_speech')


In [16]:
train_dataset = tokenize_and_create_dataset(tokenizer, train_df, max_length)

barishal_valid_dataset   = tokenize_and_create_dataset(tokenizer, barishal_valid_df,   max_length)
chittagong_valid_dataset = tokenize_and_create_dataset(tokenizer, chittagong_valid_df, max_length)
mymensingh_valid_dataset = tokenize_and_create_dataset(tokenizer, mymensingh_valid_df, max_length)
noakhali_valid_dataset   = tokenize_and_create_dataset(tokenizer, noakhali_valid_df,   max_length)
sylhet_valid_dataset     = tokenize_and_create_dataset(tokenizer, sylhet_valid_df,     max_length)
standard_valid_dataset   = tokenize_and_create_dataset(tokenizer, standard_valid_df,   max_length)

barishal_test_dataset    = tokenize_and_create_dataset(tokenizer, barishal_test_df,    max_length)
chittagong_test_dataset  = tokenize_and_create_dataset(tokenizer, chittagong_test_df,  max_length)
mymensingh_test_dataset  = tokenize_and_create_dataset(tokenizer, mymensingh_test_df,  max_length)
noakhali_test_dataset    = tokenize_and_create_dataset(tokenizer, noakhali_test_df,    max_length)
sylhet_test_dataset      = tokenize_and_create_dataset(tokenizer, sylhet_test_df,      max_length)
standard_test_dataset    = tokenize_and_create_dataset(tokenizer, standard_test_df,    max_length)



=== First 3 tokenized examples ===
  [0] Input: লকট মোর প্রিয় ফল
       Label: Jamrul is my favorite fruit
  [1] Input: যত হারাহারি সম্ভব বিয়ার ব্যবস্থা গরো
       Label: Arrange marriage as soon as possible
  [2] Input: যত তাড়াতাড়ি সম্ভব বিয়ার ব্যবস্থা কর
       Label: Arrange marriage as soon as possible

=== First 3 tokenized examples ===
  [0] Input: বাংলাদেশে ৬৪ ডা জেলা
       Label: 64 districts in Bangladesh
  [1] Input: বরিশালের মানু ক্যামন হয়?
       Label: How are the people of Barisal?
  [2] Input: খুলনা জেলা কি বেমালা সুন্দর?
       Label: Khulna district is very beautiful?

=== First 3 tokenized examples ===
  [0] Input: বাংলাদেশত ৬৪ ইয়ান জেলা
       Label: 64 districts in Bangladesh
  [1] Input: বরিশালর মানুষ হইল্লে অয় দে?
       Label: How are the people of Barisal?
  [2] Input: খুলনা জেলা কি বহুত সুন্দর নাকি?
       Label: Khulna district is very beautiful?

=== First 3 tokenized examples ===
  [0] Input: বাংলাদেশে ৬৪ টা জেলা
       Label: 64 districts in Bangl

In [17]:
import sacrebleu

def decode_strip_pad(tokenizer, sequences):
    out = []
    for seq in sequences:
        seq = [t for t in seq if t not in [tokenizer.pad_token_id, -100]]
        out.append(tokenizer.decode(seq, skip_special_tokens=True,
                                   clean_up_tokenization_spaces=True))
    return out

def normalize_texts(texts):
    return [' '.join(t.strip().split()) for t in texts]

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds  = normalize_texts(decode_strip_pad(tokenizer, preds))
    decoded_labels = normalize_texts(decode_strip_pad(tokenizer, labels))
    bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels], tokenize='13a').score
    return {'bleu': bleu}


In [18]:
from transformers import TrainerState, TrainerControl

class MultiValidationBLEUCallback(TrainerCallback):
    def __init__(self, val_datasets, trainer, compute_metrics_fn, tokenizer):
        self.val_datasets       = val_datasets
        self.trainer            = trainer
        self.compute_metrics_fn = compute_metrics_fn
        self.tokenizer          = tokenizer

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f'\n=== BLEU Evaluation at epoch {state.epoch:.2f} ===')
        for name, dataset in self.val_datasets.items():
            outputs = self.trainer.predict(dataset)
            result  = self.compute_metrics_fn((outputs.predictions, outputs.label_ids))
            print(f'  {name}: BLEU = {result["bleu"]:.2f}')
            preds   = decode_strip_pad(self.tokenizer, outputs.predictions[:5])
            refs    = decode_strip_pad(self.tokenizer, outputs.label_ids[:5])
            srcs    = decode_strip_pad(self.tokenizer, dataset[:5]['input_ids'])
            for i in range(len(preds)):
                print(f'  Source    : {srcs[i]}')
                print(f'  Reference : {refs[i]}')
                print(f'  Prediction: {preds[i]}')
                print('  ' + '-' * 48)
        return control


In [19]:
from datasets import concatenate_datasets

val_datasets = {
    'Barishal':   barishal_valid_dataset,
    'Chittagong': chittagong_valid_dataset,
    'Mymensingh': mymensingh_valid_dataset,
    'Noakhali':   noakhali_valid_dataset,
    'Sylhet':     sylhet_valid_dataset,
    'Standard':   standard_valid_dataset,
}

eval_dataset = concatenate_datasets(list(val_datasets.values()))


In [20]:
unfreeze_schedule = {
    1: 2,      # unfreeze top 2 layers at epoch 1
    2: 4,      # unfreeze top 4 layers at epoch 2
    3: 'all',  # unfreeze all at epoch 3
}

gradual_unfreeze_cb = GradualUnfreezingCallback(
    model=model, unfreeze_schedule=unfreeze_schedule
)


In [21]:
# forced_bos_token_id ensures English output during generate()
forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG]
print(f"forced_bos_token_id for '{TGT_LANG}': {forced_bos_token_id}")

model_args = Seq2SeqTrainingArguments(
    output_dir='./output_dir',
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    num_train_epochs=6,
    weight_decay=0.01,
    lr_scheduler_type='linear',
    eval_strategy='epoch',
    logging_steps=200,
    save_strategy='epoch',
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    predict_with_generate=True,
    # mBART-50: pass forced_bos_token_id via generation config
    generation_max_length=max_length,
    generation_num_beams=4,
    fp16=True,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=model_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    callbacks=[gradual_unfreeze_cb],
)

# Patch model.generate to always include forced_bos_token_id
import functools
_orig_generate = model.generate
model.generate = functools.partial(_orig_generate, forced_bos_token_id=forced_bos_token_id)

# Multi-validation BLEU callback
multi_val_bleu_cb = MultiValidationBLEUCallback(
    val_datasets=val_datasets,
    trainer=trainer,
    compute_metrics_fn=compute_metrics,
    tokenizer=tokenizer
)
trainer.add_callback(multi_val_bleu_cb)

trainer.train()

forced_bos_token_id for 'en_XX': 250004


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Bleu
1,9.273240,1.186047,28.970973
2,4.756340,1.474809,28.030313
3,1.690251,1.486894,31.212599
4,1.250490,1.660841,28.710823
5,0.700679,1.681458,30.009841
6,0.577348,1.715989,31.603611



=== BLEU Evaluation at epoch 1.00 ===
  Barishal: BLEU = 32.67
  Source    : বাংলাদেশে ৬৪ ডা জেলা
  Reference : 64 districts in Bangladesh
  Prediction: Today's weather is very good
  ------------------------------------------------
  Source    : বরিশালের মানু ক্যামন হয়?
  Reference : How are the people of Barisal?
  Prediction: What is the name of the village?
  ------------------------------------------------
  Source    : খুলনা জেলা কি বেমালা সুন্দর?
  Reference : Khulna district is very beautiful?
  Prediction: What other districts are you in?
  ------------------------------------------------
  Source    : কহনো ভাইববা দ্যাখছো মুই না থাকলে তোমার কি হইতে?
  Reference : Have you ever wondered what would happen to you without me?
  Prediction: What to do if you don't like it?
  ------------------------------------------------
  Source    : এতো ভাবার সময় কোম্মে?
  Reference : Where is the time to think so much?
  Prediction: When is the time for thought?
  --------------------------

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


=== BLEU Evaluation at epoch 5.00 ===
  Barishal: BLEU = 29.89
  Source    : বাংলাদেশে ৬৪ ডা জেলা
  Reference : 64 districts in Bangladesh
  Prediction: South District Cox's Bazar
  ------------------------------------------------
  Source    : বরিশালের মানু ক্যামন হয়?
  Reference : How are the people of Barisal?
  Prediction: What is the meaning of this prayer?
  ------------------------------------------------
  Source    : খুলনা জেলা কি বেমালা সুন্দর?
  Reference : Khulna district is very beautiful?
  Prediction: Has the electricity problem been solved?
  ------------------------------------------------
  Source    : কহনো ভাইববা দ্যাখছো মুই না থাকলে তোমার কি হইতে?
  Reference : Have you ever wondered what would happen to you without me?
  Prediction: What to do if a brother or sister is not alive?
  ------------------------------------------------
  Source    : এতো ভাবার সময় কোম্মে?
  Reference : Where is the time to think so much?
  Prediction: When is the time?
  --------------

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== BLEU Evaluation at epoch 6.00 ===
  Barishal: BLEU = 30.92
  Source    : বাংলাদেশে ৬৪ ডা জেলা
  Reference : 64 districts in Bangladesh
  Prediction: South District Cox's Bazar
  ------------------------------------------------
  Source    : বরিশালের মানু ক্যামন হয়?
  Reference : How are the people of Barisal?
  Prediction: Have you ever visited Bangladesh?
  ------------------------------------------------
  Source    : খুলনা জেলা কি বেমালা সুন্দর?
  Reference : Khulna district is very beautiful?
  Prediction: Has the electricity problem been solved?
  ------------------------------------------------
  Source    : কহনো ভাইববা দ্যাখছো মুই না থাকলে তোমার কি হইতে?
  Reference : Have you ever wondered what would happen to you without me?
  Prediction: What to do if not found by eating?
  ------------------------------------------------
  Source    : এতো ভাবার সময় কোম্মে?
  Reference : Where is the time to think so much?
  Prediction: When is the time to think?
  --------------------

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1758, training_loss=2.536524958171128, metrics={'train_runtime': 2649.4271, 'train_samples_per_second': 21.229, 'train_steps_per_second': 0.664, 'total_flos': 1.5236005833474048e+16, 'train_loss': 2.536524958171128, 'epoch': 6.0})

In [22]:
trainer.save_model('/content/drive/MyDrive/finetuned_mbart50')
tokenizer.save_pretrained('/content/drive/MyDrive/finetuned_mbart50')
print('Model saved to /content/drive/MyDrive/finetuned_mbart50')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/finetuned_mbart50


In [23]:
!zip -r /content/drive/MyDrive/model_checkpoints.zip /content/drive/MyDrive/output_dir


	zip warning: name not matched: /content/drive/MyDrive/output_dir

zip error: Nothing to do! (try: zip -r /content/drive/MyDrive/model_checkpoints.zip . -i /content/drive/MyDrive/output_dir)


In [24]:
from IPython.display import display, HTML
display(HTML('<a href="/content/drive/MyDrive/model_checkpoints.zip" target="_blank">Download Model Checkpoints</a>'))


## Inference & Final BLEU Evaluation


In [30]:
import torch, pandas as pd, sacrebleu, time, os
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration
from tqdm import tqdm
from normalizer import normalize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load fine-tuned model
MODEL_PATH = '/content/drive/MyDrive/finetuned_mbart50'
inf_tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_PATH, src_lang='bn_IN', tgt_lang='en_XX')
inf_model     = MBartForConditionalGeneration.from_pretrained(MODEL_PATH).to(device)
inf_model.eval()

# forced_bos_token_id is essential for mBART-50 to generate English
forced_bos = inf_tokenizer.lang_code_to_id['en_XX']
print(f'forced_bos_token_id: {forced_bos}')

def translate_batch(texts, model, tokenizer, device, batch_size=8, max_length=128, num_beams=5):
    results = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Translating', unit='batch'):
        batch = [normalize(str(x).strip()) for x in texts[i:i+batch_size]]
        inputs = tokenizer(batch, return_tensors='pt', padding=True,
                           truncation=True, max_length=max_length).to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos,   # mBART-50 requirement
                max_length=max_length,
                num_beams=num_beams
            )
        results.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return results

def evaluate_dialect(name, csv_path, bangla_col):
    print(f"\n{'='*60}\n Evaluating: {name}\n{'='*60}")
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    col = bangla_col.strip()
    if col not in df.columns:
        fallback = [c for c in df.columns if 'bangla' in c.lower()]
        col = fallback[0] if fallback else None
    if not col:
        print('  No Bangla column found. Skipping.')
        return None
    sources = df[col].astype(str).tolist()
    refs    = df['english_speech'].astype(str).tolist()
    t0 = time.time()
    preds = translate_batch(sources, inf_model, inf_tokenizer, device)
    elapsed = time.time() - t0
    bleu = sacrebleu.corpus_bleu(preds, [refs], tokenize='13a').score
    exact = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    print(f'  Exact: {exact}/{len(preds)}')
    print(f'  BLEU: {bleu:.2f}  |  Time: {elapsed:.1f}s')
    print('\n  --- First 5 examples ---')
    for i in range(min(5, len(preds))):
        print(f'  [{i+1}] Source    : {sources[i]}')
        print(f'       Prediction: {preds[i]}')
        print(f'       Reference : {refs[i]}')
    return bleu

dialects = {
    'Barishal_':   (barishal_test_csv,   'barishal_bangla_speech '),
    'Chittagong_': (chittagong_test_csv, 'chittagong_bangla_speech '),
    'Mymensingh_': (mymensingh_test_csv, 'mymensingh_bangla_speech '),
    'Noakhali_':   (noakhali_test_csv,   'noakhali_bangla_speech '),
    'Sylhet_':     (sylhet_test_csv,     'sylhet_bangla_speech '),
    'Standard':    (sylhet_test_csv,     'bangla_speech'),
}

results = {}
for name, (path, col) in dialects.items():
    bleu = evaluate_dialect(name, path, col)
    if bleu is not None:
        results[name] = bleu

print('\n' + '='*60)
print('FINAL BLEU SCORES')
print('='*60)
for name, bleu in results.items():
    print(f'  {name:<14}: BLEU = {bleu:.2f}')

Device: cuda


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

forced_bos_token_id: 250004

 Evaluating: Barishal_


Translating: 100%|██████████| 17/17 [00:09<00:00,  1.79batch/s]


  Exact: 4/131
  BLEU: 14.96  |  Time: 9.5s

  --- First 5 examples ---
  [1] Source    : তোমার আব্বায় ক্যামন আছে?
       Prediction: Are you alive?
       Reference : How is your father?
  [2] Source    : মোর বড় বুইনের আইজগো মন ভালো নাই
       Prediction: My elder sister has no good heart
       Reference : My elder sister is not feeling well today
  [3] Source    : তুমি কি মোরে এই কামডা কইররা দেতে পারবা?
       Prediction: Can you give me a present?
       Reference : Can you do this for me?
  [4] Source    : তোমার নাহান খারাপ পোলা মুই আর এউক্কাও দেহিনাই
       Prediction: I will accept all your words and words
       Reference : I've never seen a bad boy like you
  [5] Source    : বিয়ার লইজ্ঞা মায় মোর লইজ্ঞা পোলা খোজতেয়াছে
       Prediction: Asking for our help in finding that boy
       Reference : Amma is looking for a boy for me for marriage

 Evaluating: Chittagong_


...

  Exact: 2/133
  BLEU: 10.62  |  Time: 9.1s

  --- First 5 examples ---
  [1] Source    : তোর আব্বা কেক্কা আছে?
       Prediction: Who is in your family?
       Reference : How is your father?
  [2] Source    : আর বড় আফার আইজ্জা মন ভালা নাই
       Prediction: I am not thinking about anything in class today
       Reference : My elder sister is not feeling well today
  [3] Source    : তুই কি আরে এই কাম আন করি দিতা হাইরবা নি?
       Prediction: Do you want to see this again?
       Reference : Can you do this for me?
  [4] Source    : তোর মতো শয়তান হোলা আই এইগগাও দেই নো
       Prediction: I can't accept your love
       Reference : I've never seen a bad boy like you
  [5] Source    : বিয়ার লাই আর আম্মা আর লাই হোলা টোগার
       Prediction: Cats are very comfortable animals
       Reference : Amma is looking for a boy for me for marriage

 Evaluating: Sylhet_


Translating: 100%|██████████| 17/17 [00:13<00:00,  1.30batch/s]


  Exact: 7/136
  BLEU: 16.94  |  Time: 13.0s

  --- First 5 examples ---
  [1] Source    : তোমার আব্বা বালা আছইন নি?
       Prediction: Don't you like talking to me?
       Reference : How is your father?
  [2] Source    : আমার বড় বইনর আইজ মন ভালা নায়
       Prediction: My elder sister loves me a lot
       Reference : My elder sister is not feeling well today
  [3] Source    : তুমি কিতা মোরে এই কাজটা করাই দিতা পারবা নি?
       Prediction: Can you cook in the kitchen?
       Reference : Can you do this for me?
  [4] Source    : তোমার মত খারাফ পুয়া মুই আর একটাও দেখছি না
       Prediction: I don't see your face anymore
       Reference : I've never seen a bad boy like you
  [5] Source    : বিয়ার লাগি মাই মুর লাগি ফুয়া খুজিরা
       Prediction: Boy's condition is not so good at all
       Reference : Amma is looking for a boy for me for marriage

 Evaluating: Standard


Translating: 100%|██████████| 17/17 [00:09<00:00,  1.87batch/s]

  Exact: 12/136
  BLEU: 27.00  |  Time: 9.1s

  --- First 5 examples ---
  [1] Source    : তোমার আব্বু কেমন আছে?
       Prediction: How is your father?
       Reference : How is your father?
  [2] Source    : আমার বড়  বোনের আজকে মন ভালো নেই
       Prediction: My elder sister is not feeling well today
       Reference : My elder sister is not feeling well today
  [3] Source    : তুমি কি আমাকে এই কাজটি করে দিতে পারবে?
       Prediction: Will you let me go?
       Reference : Can you do this for me?
  [4] Source    : তোমার মত খারাপ ছেলে আমি আর একটাও  দেখিনি
       Prediction: I didn't see anything wrong with you
       Reference : I've never seen a bad boy like you
  [5] Source    : বিয়ের জন্য আম্মা  আমার জন্য ছেলে  খুজতাসে
       Prediction: When my husband has a problem, he asks me to solve it
       Reference : Amma is looking for a boy for me for marriage

FINAL BLEU SCORES
  Barishal_     : BLEU = 14.96
  Chittagong_   : BLEU = 8.72
  Mymensingh_   : BLEU = 18.06
  Noakhali_     : BL

In [27]:
!pip install evaluate jiwer rouge_score
import nltk
nltk.download("wordnet")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("omw-1.4")

import evaluate
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize

cer_metric   = evaluate.load("cer")
wer_metric   = evaluate.load("wer")
rouge_metric = evaluate.load("rouge")

print("Metrics loaded.")

Metrics loaded.


In [31]:
def compute_all_metrics(predictions, references, dialect_name):
    # BLEU
    bleu = sacrebleu.corpus_bleu(predictions, [references], tokenize="13a").score

    # CER
    cer = cer_metric.compute(predictions=predictions, references=references)

    # WER
    wer = wer_metric.compute(predictions=predictions, references=references)

    # ROUGE
    rouge = rouge_metric.compute(predictions=predictions, references=references)

    # METEOR
    meteor_scores = [
        meteor_score([word_tokenize(ref)], word_tokenize(pred))
        for ref, pred in zip(references, predictions)
    ]
    meteor = sum(meteor_scores) / len(meteor_scores)

    print(f"\n{'='*55}")
    print(f"  Results: {dialect_name}")
    print(f"{'='*55}")
    print(f"  BLEU   : {bleu:.4f}")
    print(f"  CER    : {cer:.4f}  (lower is better)")
    print(f"  WER    : {wer:.4f}  (lower is better)")
    print(f"  ROUGE-1: {rouge['rouge1']:.4f}")
    print(f"  ROUGE-2: {rouge['rouge2']:.4f}")
    print(f"  ROUGE-L: {rouge['rougeL']:.4f}")
    print(f"  METEOR : {meteor:.4f}")
    print(f"{'='*55}")

    return {
        "dialect": dialect_name,
        "BLEU":    round(bleu, 4),
        "CER":     round(cer, 4),
        "WER":     round(wer, 4),
        "ROUGE-1": round(rouge["rouge1"], 4),
        "ROUGE-2": round(rouge["rouge2"], 4),
        "ROUGE-L": round(rouge["rougeL"], 4),
        "METEOR":  round(meteor, 4),
    }

In [32]:
all_results = []

for name, (path, col) in dialects.items():
    print(f"\nProcessing: {name}")

    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    c = col.strip()

    if c not in df.columns:
        fallback = [x for x in df.columns if "bangla" in x.lower()]
        c = fallback[0] if fallback else None
    if not c:
        print(f"  Skipping {name} — no Bangla column found")
        continue

    sources    = df[c].astype(str).tolist()
    references = df["english_speech"].astype(str).tolist()
    predictions = translate_batch(sources, inf_model, inf_tokenizer, device)

    result = compute_all_metrics(predictions, references, name)
    all_results.append(result)

# ── Summary table ─────────────────────────────────────────────────────────────
summary_df = pd.DataFrame(all_results).set_index("dialect")
print("\n\n" + "="*55)
print("       FINAL SUMMARY — ALL DIALECTS")
print("="*55)
print(summary_df.to_string())

# Save to Drive
summary_df.to_csv("/content/drive/MyDrive/vashantor/evaluation_results.csv")
print("\nSaved: evaluation_results.csv")


Processing: Barishal_


Translating: 100%|██████████| 17/17 [00:09<00:00,  1.80batch/s]



  Results: Barishal_
  BLEU   : 14.9620
  CER    : 0.6431  (lower is better)
  WER    : 0.8190  (lower is better)
  ROUGE-1: 0.3841
  ROUGE-2: 0.2115
  ROUGE-L: 0.3649
  METEOR : 0.3386

Processing: Chittagong_


Translating: 100%|██████████| 17/17 [00:13<00:00,  1.28batch/s]



  Results: Chittagong_
  BLEU   : 8.7182
  CER    : 0.7242  (lower is better)
  WER    : 0.9069  (lower is better)
  ROUGE-1: 0.2897
  ROUGE-2: 0.1336
  ROUGE-L: 0.2810
  METEOR : 0.2529

Processing: Mymensingh_


...

Translating: 100%|██████████| 17/17 [00:10<00:00,  1.68batch/s]



  Results: Sylhet_
  BLEU   : 16.9392
  CER    : 0.6384  (lower is better)
  WER    : 0.8113  (lower is better)
  ROUGE-1: 0.4056
  ROUGE-2: 0.2285
  ROUGE-L: 0.3870
  METEOR : 0.3638

Processing: Standard


Translating: 100%|██████████| 17/17 [00:09<00:00,  1.82batch/s]



  Results: Standard
  BLEU   : 26.9998
  CER    : 0.5023  (lower is better)
  WER    : 0.6352  (lower is better)
  ROUGE-1: 0.5253
  ROUGE-2: 0.3445
  ROUGE-L: 0.5114
  METEOR : 0.4882


       FINAL SUMMARY — ALL DIALECTS
                BLEU     CER     WER  ROUGE-1  ROUGE-2  ROUGE-L  METEOR
dialect                                                                
Barishal_    14.9620  0.6431  0.8190   0.3841   0.2115   0.3649  0.3386
Chittagong_   8.7182  0.7242  0.9069   0.2897   0.1336   0.2810  0.2529
Mymensingh_  18.0590  0.6005  0.7679   0.4328   0.2437   0.4151  0.3812
Noakhali_    10.6179  0.7014  0.8906   0.3090   0.1437   0.2937  0.2599
Sylhet_      16.9392  0.6384  0.8113   0.4056   0.2285   0.3870  0.3638
Standard     26.9998  0.5023  0.6352   0.5253   0.3445   0.5114  0.4882


OSError: Cannot save file into a non-existent directory: '/content/drive/MyDrive/vashantor'